# SIB-200 baseline on Colab

Full fine-tuning and LoRA (r=8) on `Davlan/afro-xlmr-large-76L`, 6 languages (amh, hau, kin, lug, swa, zul), 3 seeds.
Each run trains on one source language and tests on all 6, so the 6x6 zero-shot transfer matrix comes from the same 36 runs.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all.

**If Colab disconnects:** reconnect and Run all again. Results are written to Google Drive after every run,
and the script skips the runs that already finished, so nothing is lost except the run that was in progress.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run again."

## 2. Mount Google Drive (results are saved here)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/typology-aware-peft")
OUT = DRIVE / "results" / "sib200"
OUT.mkdir(parents=True, exist_ok=True)
print("Results folder:", OUT)

## 3. Get the scripts

Tries, in order: the GitHub repo, a copy saved on Drive by an earlier session, then asks you to upload
`sib200_baseline.py` and `sib200_report.py` from the `src` folder of the repo.

In [ ]:
import os, re, shutil
REPO_URL = "https://github.com/WanjaWhoopie/typology-aware-peft.git"
WORK = Path("/content/typology-aware-peft")
SCRIPTS = ["sib200_baseline.py", "sib200_report.py"]
SAVED = DRIVE / "src"

if not WORK.exists():
    !git clone -q {REPO_URL} {WORK}
(WORK / "src").mkdir(parents=True, exist_ok=True)

for name in SCRIPTS:
    if not (WORK / "src" / name).exists() and (SAVED / name).exists():
        shutil.copy(SAVED / name, WORK / "src" / name)

missing = [n for n in SCRIPTS if not (WORK / "src" / n).exists()]
if missing:
    print("Not on GitHub (not pushed yet, or the repo is private). Please upload:", missing)
    from google.colab import files
    for name, content in files.upload().items():
        name = re.sub(r" \(\d+\)", "", name)  # browsers rename repeat uploads to "x (1).py"
        (WORK / "src" / name).write_bytes(content)

missing = [n for n in SCRIPTS if not (WORK / "src" / n).exists()]
assert not missing, f"Still missing: {missing}"
SAVED.mkdir(parents=True, exist_ok=True)
for name in SCRIPTS:  # keep a copy on Drive so a reconnect does not need another upload
    shutil.copy(WORK / "src" / name, SAVED / name)
os.chdir(WORK)
print("Working in", os.getcwd())

## 4. Install packages

Versions are pinned to the ones the smoke test passed with, so results are reproducible.

In [ ]:
!pip install -q transformers==4.57.2 datasets==4.4.1 peft==0.21.2 scikit-learn
import transformers, datasets, peft
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| peft", peft.__version__)

## 5. Smoke test (about 2 minutes)

LoRA on the base model, 2 languages, 64 examples, 1 epoch. Only checks that everything runs;
the accuracy is not meaningful and a "collapsed" warning here is normal.

In [ ]:
!python src/sib200_baseline.py --smoke --model Davlan/afro-xlmr-base

## 6. Full run (roughly 2 to 4 hours on a T4)

36 runs: full fine-tuning first, then LoRA. Each line ends with an estimate of the time left.

* **Out of memory** during full fine-tuning: rerun this cell with `--bs 8` added
  (and write down that the batch size changed).
* **Collapsed run** (warning: dev accuracy below 0.3): finish the job, then use step 8.

In [ ]:
!python src/sib200_baseline.py --out {OUT}

## 7. Check the runs

In [ ]:
import json
runs = [json.loads(l) for l in (OUT / "runs.jsonl").read_text().splitlines() if l.strip()]
print(f"{len(runs)} of 36 runs finished\n")
print(f"{'method':6} {'seed':>4} {'src':4} {'dev acc':>7} {'epoch':>5} {'min':>5} {'GB':>5}  collapsed")
for r in sorted(runs, key=lambda r: (r["method"], r["src"], r["seed"])):
    print(f"{r['method']:6} {r['seed']:>4} {r['src']:4} {r['best_dev_acc']:>7.3f} {r['best_epoch']:>5} "
          f"{r['seconds'] / 60:>5.1f} {r['peak_mem_gb'] or 0:>5.1f}  {'YES' if r['collapsed'] else ''}")

## 8. Optional: redo collapsed runs with a lower learning rate

Only if step 7 shows collapsed runs. Set `REDO = True`, run this cell, then run step 7 again.
It removes the collapsed runs from the results files and trains them again with half the learning rate.
The new learning rate is stored in `runs.jsonl`, so mention it in the write-up.

In [ ]:
REDO = False

def key(r):
    return (r["method"], r["seed"], r["src"])

if REDO:
    bad = {key(r) for r in runs if r["collapsed"]}
    print("Redoing:", sorted(bad))
    for fname in ("runs.jsonl", "results.jsonl"):
        f = OUT / fname
        keep = [l for l in f.read_text().splitlines() if l.strip() and key(json.loads(l)) not in bad]
        f.write_text("\n".join(keep) + "\n")
    METHODS = " ".join(sorted({m for m, _, _ in bad}))
    if METHODS:
        !python src/sib200_baseline.py --out {OUT} --methods {METHODS} --lr_full 1e-5 --lr_lora 1.5e-4

## 9. Report

In [ ]:
!python src/sib200_report.py --dir {OUT}
from IPython.display import Markdown, display
display(Markdown((OUT / "summary.md").read_text()))

## 10. Download the results

Unzip into `results/sib200/` in the repo on your laptop. The same files also stay on Google Drive
in `typology-aware-peft/results/sib200`.

In [ ]:
shutil.make_archive("/content/sib200_results", "zip", OUT)
from google.colab import files
files.download("/content/sib200_results.zip")